# 🎾 Tennis Analysis — Colab training

Trains the three models the pipeline needs. **Run Section 0 first**, then jump to
whichever model you want — the sections are independent.

| Section | Model | ~T4 time |
|---|---|---|
| 1 | Ball detector (YOLO11) | 20–40 min |
| 2 | Court keypoints (ResNet18) | 15–30 min |
| 3 | Stroke classifier (RF / 1D-CNN) | 30–90 min (pose extraction dominates) |

**Before you start:** `Runtime → Change runtime type → T4 GPU`. Free tier gives ~15–30
GPU hours/week and sessions up to ~12 h, but idle sessions are reclaimed after ~90 min —
so every section writes its weights to Google Drive as soon as they exist.

## Section 0 — Setup

Run all four cells.

In [ ]:
# Confirm a GPU is actually attached — this is the #1 cause of "why is it so slow".
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
      or "NO GPU — set Runtime > Change runtime type > T4 GPU")
print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}")

In [ ]:
# Mount Drive. Weights land here so a disconnected session doesn't lose the run.
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/tennis_analysis")
(DRIVE / "models").mkdir(parents=True, exist_ok=True)
print("artefacts ->", DRIVE / "models")

In [ ]:
# --- Get the code into Colab ---------------------------------------------------
# Option A (recommended): push the project to GitHub, then clone it.
REPO = ""   # e.g. "https://github.com/yourname/tennis_analysis.git"

import os, shutil
from pathlib import Path

PROJECT = Path("/content/tennis_analysis")
if REPO:
    if PROJECT.exists():
        shutil.rmtree(PROJECT)
    !git clone -q $REPO $PROJECT
else:
    # Option B: zip the project locally, upload it here.
    #   On your Mac:  zip -r tennis_analysis.zip . -x '.venv/*' 'output/*' 'data/*'
    from google.colab import files
    if not PROJECT.exists():
        up = files.upload()
        name = next(iter(up))
        !unzip -q $name -d /content/tennis_analysis_unzipped
        inner = next(p for p in Path("/content/tennis_analysis_unzipped").rglob("main.py")).parent
        shutil.move(str(inner), str(PROJECT))

os.chdir(PROJECT)
print("cwd:", Path.cwd())
assert (PROJECT / "main.py").exists(), "project not found — check REPO or your upload"

In [ ]:
# Install only what Colab lacks. Deliberately NOT installing torch/torchvision:
# Colab ships a CUDA-matched build, and reinstalling it is the classic way to
# end up with a torch that can't see the GPU.
!pip install -q ultralytics roboflow pyyaml joblib scikit-learn

import torch
print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}  <- must still be True")

---
## Section 1 — Ball detector

Dataset: [viren-dhanwani/tennis-ball-detection](https://universe.roboflow.com/viren-dhanwani/tennis-ball-detection)
(578 images). Free API key: app.roboflow.com → Settings → API key.

In [ ]:
# Prompted, not pasted: a literal key typed here gets saved into the .ipynb and
# would be committed to git. getpass keeps it in memory only.
import os, getpass

try:  # Colab's built-in secret store (key icon in the sidebar), if you've set it up
    from google.colab import userdata
    os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    os.environ["ROBOFLOW_API_KEY"] = getpass.getpass("Roboflow API key: ")

assert os.environ["ROBOFLOW_API_KEY"], "set your Roboflow API key first"
print("key loaded")

In [ ]:
!python scripts/download_datasets.py --dataset ball --output data

# Sanity-check what actually landed before burning GPU time on it.
from pathlib import Path
import yaml
cfg = next(Path("data/ball").rglob("data.yaml"))
print("data.yaml:", cfg)
print(yaml.safe_load(cfg.read_text()))
for split in ("train", "valid", "test"):
    imgs = list(Path("data/ball").rglob(f"{split}/images/*"))
    print(f"  {split}: {len(imgs)} images")

In [ ]:
# imgsz=1280 is the single biggest lever here: a tennis ball is often <10px wide at
# 640 and simply vanishes. Costs ~3x the time, worth it.
!python scripts/train_ball_detector.py \
    --data $(find data/ball -name data.yaml | head -1) \
    --model yolo11n.pt \
    --epochs 60 \
    --imgsz 1280 \
    --batch 8 \
    --device 0 \
    --output models/ball_yolo11n.pt

In [ ]:
# Verify the weights load through the real inference class, then persist to Drive.
import shutil
from tennis_analysis.config import BallConfig
from tennis_analysis.detection import BallDetector

detector = BallDetector(BallConfig(model="models/ball_yolo11n.pt", confidence=0.15))
print("ball detector loads OK")

shutil.copy2("models/ball_yolo11n.pt", DRIVE / "models" / "ball_yolo11n.pt")
print("saved ->", DRIVE / "models" / "ball_yolo11n.pt")

---
## Section 2 — Court keypoints

⚠️ **Read this before training.** The model regresses 14 keypoints in the canonical
order defined by `KEYPOINT_NAMES` in `tennis_analysis/court_keypoints/geometry.py`.
Public datasets vary in both *count* and *order*. The inspection cell below tells you
what you actually downloaded — if the order differs, remap it with `--keypoint-order`
rather than training a model that predicts scrambled points.

In [ ]:
!python scripts/download_datasets.py --dataset court --output data

In [ ]:
# --- Inspect the annotations BEFORE training -----------------------------------
import json
from pathlib import Path
from tennis_analysis.court_keypoints.geometry import KEYPOINT_NAMES, NUM_KEYPOINTS

ann_files = sorted(Path("data/court").rglob("_annotations.coco.json"))
print("annotation files:", [str(p) for p in ann_files] or "NONE FOUND")

if ann_files:
    payload = json.loads(ann_files[0].read_text())
    anns = payload.get("annotations", [])
    print(f"\nimages={len(payload.get('images', []))}  annotations={len(anns)}")

    counts = {len(a.get("keypoints", [])) // 3 for a in anns if a.get("keypoints")}
    print(f"keypoints per annotation: {counts or 'NO keypoints field!'}")
    print(f"this project expects: {NUM_KEYPOINTS}")

    for cat in payload.get("categories", []):
        if cat.get("keypoints"):
            print("\nsource keypoint names:")
            for i, n in enumerate(cat["keypoints"]):
                print(f"  {i:2d}  {n}")

    print("\ncanonical order this repo expects:")
    for i, n in enumerate(KEYPOINT_NAMES):
        print(f"  {i:2d}  {n}")

In [ ]:
# --- Visualise one annotated sample --------------------------------------------
# Compare the drawn indices against the canonical order printed above. If they
# disagree, build a permutation and pass it via --keypoint-order below.
import json, cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

ann_file = sorted(Path("data/court").rglob("_annotations.coco.json"))[0]
payload = json.loads(ann_file.read_text())
images = {im["id"]: im for im in payload["images"]}

sample = next(a for a in payload["annotations"] if a.get("keypoints"))
meta = images[sample["image_id"]]
img = cv2.imread(str(ann_file.parent / meta["file_name"]))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

kps = np.array(sample["keypoints"], dtype=float).reshape(-1, 3)
plt.figure(figsize=(13, 8))
plt.imshow(img)
for i, (x, y, v) in enumerate(kps):
    if v > 0:
        plt.scatter([x], [y], c="lime", s=45, edgecolors="black", zorder=3)
        plt.annotate(str(i), (x, y), color="yellow", fontsize=13, weight="bold",
                     xytext=(6, 6), textcoords="offset points")
plt.title(f"{meta['file_name']} — {int((kps[:, 2] > 0).sum())} visible keypoints")
plt.axis("off"); plt.show()

In [ ]:
# If the source order matches the canonical order, leave ORDER empty.
# Otherwise ORDER[canonical_index] = source_index, e.g. "0,1,3,2,4,5,..." (14 values).
ORDER = ""

order_flag = f"--keypoint-order {ORDER}" if ORDER else ""
!python scripts/train_court_keypoints.py \
    --data data/court \
    --epochs 60 \
    --batch 32 \
    --device 0 \
    $order_flag \
    --output models/court_keypoints_resnet18.pt

In [ ]:
# Watch mean_kp_err in the log above — that's the number that matters.
# Under ~5 px on a 224 input is usable; over ~15 px the homography will be unreliable.
import shutil
from tennis_analysis.config import CourtConfig
from tennis_analysis.court_keypoints.detector import CourtKeypointDetector

detector = CourtKeypointDetector(CourtConfig(model="models/court_keypoints_resnet18.pt"))
print("court model loads OK")

shutil.copy2("models/court_keypoints_resnet18.pt",
             DRIVE / "models" / "court_keypoints_resnet18.pt")
print("saved ->", DRIVE / "models" / "court_keypoints_resnet18.pt")

---
## Section 3 — Stroke classifier

THETIS is gated behind a request form at
[thetis.image.ece.ntua.gr](http://thetis.image.ece.ntua.gr/) — it cannot be scripted.
Once you have it, upload the archive to Drive at
`MyDrive/tennis_analysis/thetis.zip` and run the cells below.

Expected layout after extraction — one directory per source class:

```
data/thetis/
  forehand_flat/    *.avi
  backhand_slice/   *.avi
  service_flat/     *.avi
  forehand_volley/  *.avi
```

In [ ]:
# Extract THETIS from Drive.
from pathlib import Path
archive = DRIVE / "thetis.zip"
assert archive.exists(), f"upload THETIS to {archive} first"

Path("data").mkdir(exist_ok=True)
!unzip -q -o "$archive" -d data/thetis

for d in sorted(p for p in Path("data/thetis").rglob("*") if p.is_dir()):
    clips = [c for c in d.glob("*") if c.suffix.lower() in {".avi", ".mp4", ".mov"}]
    if clips:
        print(f"{d.relative_to('data/thetis')}: {len(clips)} clips")

In [ ]:
# Check how the source directories fold into the four stroke labels.
!python scripts/train_stroke_classifier.py --show-label-map

In [ ]:
# Pose extraction dominates runtime and is the part worth caching: --cache writes the
# extracted poses to Drive, so retraining or switching backends later is instant.
!python scripts/train_stroke_classifier.py \
    --data data/thetis \
    --backend random_forest \
    --cache "$DRIVE/thetis_poses.npz" \
    --output models/stroke_classifier.joblib

In [ ]:
# Same cached poses, temporal backend — compare the confusion matrices.
!python scripts/train_stroke_classifier.py \
    --data data/thetis \
    --backend cnn1d \
    --cache "$DRIVE/thetis_poses.npz" \
    --output models/stroke_classifier_cnn.pt

In [ ]:
import shutil
from pathlib import Path
from tennis_analysis.stroke_classification.classifier import load_classifier

for name in ("stroke_classifier.joblib", "stroke_classifier_cnn.pt"):
    src = Path("models") / name
    if src.exists():
        print(f"{name}: labels = {load_classifier(src).labels}")
        shutil.copy2(src, DRIVE / "models" / name)
        print(f"  saved -> {DRIVE / 'models' / name}")

---
## Section 4 — Bring the weights home

Everything is already on Drive under `MyDrive/tennis_analysis/models/`. Either sync
that folder to your Mac, or download directly with the cell below.

Then drop the files into `models/` locally and run:

```bash
python main.py --input input_videos/match.mp4 --output output/ --dashboard
```

In [ ]:
from google.colab import files
from pathlib import Path

for path in sorted((DRIVE / "models").glob("*")):
    print("downloading", path.name)
    files.download(str(path))